# Cluster Candidate Selection

This notebook identifies suitable galaxy cluster candidates for luminosity function analysis from the HeCS catalog.

## Setup and Configuration

Import necessary libraries and configure plotting parameters and cosmology (H0=70, ΩM=0.3, ΩΛ=0.7).

In [1]:
# Import necessary libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import LambdaCDM
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Set cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "bold",
    'font.size': 25,
    'font.weight': 'normal',
    
    # Tick direction and appearance
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,            # show top ticks
    'ytick.right': True,          # show right ticks
    'xtick.minor.visible': True,  # show minor x ticks
    'ytick.minor.visible': True,  # show minor y ticks
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    
    # Axes and line properties
    'lines.linewidth': 2,
    'axes.linewidth': 3.5,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})
# Pandas configuration
pd.set_option('display.max_columns', None)

## Distance Modulus and Absolute Magnitude Calculation

Calculate the absolute magnitude from apparent magnitude using cosmological distance modulus at z=0.03.

In [6]:
from astropy.cosmology import LambdaCDM
import astropy.units as u
import numpy as np

# Define cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

# Given values
z = 0.03
apparent_mag = 20.2

# (1) Compute luminosity distance (in Mpc → convert to pc)
lum_dist = cosmo.luminosity_distance(z).to(u.pc)

# (2) Compute distance modulus correctly:
distance_modulus = 5 * np.log10(lum_dist.value) - 5

# Alternatively, use astropy's built-in:
# distance_modulus = cosmo.distmod(z).value

# (3) Compute absolute magnitude
absolute_mag = apparent_mag - distance_modulus

print(f"Luminosity Distance: {lum_dist:.2e}")
print(f"Distance Modulus: {distance_modulus:.2f}")
print(f"Absolute Magnitude: {absolute_mag:.2f}")


Luminosity Distance: 1.31e+08 pc
Distance Modulus: 35.59
Absolute Magnitude: -15.39


## Load HeCS VAC Data

Load the HeCS (Hectospec Cluster Survey) Value Added Catalog containing cluster information.

In [3]:
AllHeCS_VAC = pd.read_csv('../../DATA/AllHeCS_VAC_updated.csv')

## Select Rich Clusters at z < 0.04

Filter clusters with:
- Redshift z < 0.04
- More than 100 member galaxies within R200
- Display key properties: mass (M200), radius (R200), number of members

In [4]:
AllHeCS_VAC[(AllHeCS_VAC['Z']<0.04)&(AllHeCS_VAC['NMEM_R200']>100)][['CLID', 'RA', 'DEC', 'Z', 'NMEM_R200', 'NSPEC_R200','M200','R200','R200_arcmin']]

,CLID,RA,DEC,Z,NMEM_R200,NSPEC_R200,M200,R200,R200_arcmin
1,A1367,176.175872,19.734385,0.022466,194,929.0,4.743239e+14,1.60,58.749698
4,Coma,195.000629,27.969336,0.023416,566,1673.0,6.378430e+14,1.76,62.073881
11,A2199,247.168609,39.548985,0.030956,376,1659.0,3.377935e+14,1.42,38.228744
14,A1185,167.697242,28.693497,0.033627,124,400.0,3.081018e+14,1.38,34.310543
15,A2063,230.758295,8.625421,0.034054,114,361.0,3.879547e+14,1.49,36.599636
17,A2147,240.573005,15.904449,0.036198,206,505.0,4.279491e+14,1.54,35.678658


## Alternative Selection: All Clusters at z < 0.031

Show all clusters with z < 0.031 regardless of richness.

In [5]:
AllHeCS_VAC[(AllHeCS_VAC['Z']<0.031)][['CLID', 'RA', 'DEC', 'Z', 'NMEM_R200', 'NSPEC_R200','M200','R200','R200_arcmin']]

,CLID,RA,DEC,Z,NMEM_R200,NSPEC_R200,M200,R200,R200_arcmin
0,MKW4,181.124967,1.872426,0.020430,93,844.0,1.619519e+14,1.12,45.112168
1,A1367,176.175872,19.734385,0.022466,194,929.0,4.743239e+14,1.60,58.749698
2,MKW11,202.361705,11.709481,0.023305,37,192.0,6.168133e+13,0.81,28.700320
3,A779,139.934056,33.710087,0.023240,38,140.0,4.374229e+13,0.72,25.580744
4,Coma,195.000629,27.969336,0.023416,566,1673.0,6.378430e+14,1.76,62.073881
5,NGC4325,185.753950,10.565571,0.025480,20,100.0,2.490919e+13,0.60,19.495780
6,RXCJ2214p1350,333.667017,13.853890,0.026405,29,105.0,3.832588e+13,0.69,21.658839
7,MKW8,220.158347,3.472353,0.027100,61,201.0,6.591711e+13,0.83,25.406457
8,NGC6338,258.846348,57.427045,0.028770,60,144.0,1.230218e+14,1.02,29.469129
9,A2197,247.477376,40.662700,0.029979,94,203.0,1.001133e+14,0.95,26.378123
